# Gallery — what the samples actually look like

The metric tables say these ensembles are **over-confident**: the spread is a fraction of
the error. This notebook is for seeing that rather than reading it — a collapsed cloud and
a well-spread one give very different pictures, and a reconstruction can score respectably
while being visibly wrong in a way no scalar reports.

Four views, each with its knobs at the top of its cell:

1. **One instance in depth** — truth, ensemble mean, error, uncertainty, individual samples
2. **Methods side by side** — the same instance under every method and N
3. **Several instances** — is the behaviour typical or cherry-picked?
4. **Observation space** — does the reconstruction reproduce the data?

**Colour follows the physics.** Vorticity (navier-stokes) is signed, so it gets a diverging
map centred on zero; permittivity and intensity are positive, so they get a single-hue
sequential one. Error panels are always diverging and symmetric about zero. Panels showing
the *same* quantity share one colour scale, so they can be compared by eye — that is the
whole point of a gallery, and it is lost the moment each panel is normalised separately.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('/mnt/home/gandry/kps/InverseBench')
sys.path.insert(0, str(ROOT / 'scripts'))

import matplotlib.pyplot as plt, numpy as np, torch
import report_metrics as rm

plt.rcParams.update({'figure.dpi': 110, 'font.size': 8.5, 'axes.titlesize': 8.5})

RUNS = {}                      # (problem, algo, run) -> sorted list of result files
for problem, algo, run, files in rm.runs(pattern='gen*'):
    RUNS[(problem, algo, run)] = files
for k, v in RUNS.items():
    print(f'{k[0]:<14}{k[1]:<6}{k[2]:<12}{len(v):>5} ids')

In [ ]:
def load(problem, algo, run, i):
    r"""One saved ensemble as (samples, truth), both (…, C, H, W) and unnormalised."""
    d = torch.load(RUNS[(problem, algo, run)][i], map_location='cpu', weights_only=False)
    x = d['recon'].float()
    t = d['target'].float().reshape(1, *x.shape[1:])
    return x, t, d


def scale(a, signed=None, sym=False):
    r"""Colour scale and map for a field. `signed` is detected from the data unless given:
    a quantity that takes both signs is centred on zero with a diverging map, a positive one
    gets a single-hue sequential map. `sym` forces the diverging treatment (error panels).
    """
    a = np.asarray(a)
    if signed is None:
        signed = bool(a.min() < 0 and a.max() > 0 and abs(a.min()) > 0.05 * a.max())
    if sym or signed:
        v = float(np.abs(a).max())
        return dict(cmap='RdBu_r', vmin=-v, vmax=v)
    return dict(cmap='Blues', vmin=float(a.min()), vmax=float(a.max()))


def panel(ax, img, title, cb=True, **kw):
    im = ax.imshow(np.asarray(img).squeeze(), **kw)
    ax.set_title(title, fontsize=8.5)
    ax.set_xticks([]); ax.set_yticks([])
    if cb:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    return im

print('helpers ready')

## 1. One instance in depth

The truth, what the ensemble says, and how wrong it is. **`uncertainty` and `|error|` should
look alike** — the cloud should be widest exactly where it is most wrong. Where the error map
has structure the uncertainty map does not, the ensemble is confidently wrong there.

In [ ]:
PROBLEM, ALGO, RUN, ID = 'navier-stokes', 'KPSH', 'gen_N512', 0      # <- edit
N_SHOW = 4

x, t, d = load(PROBLEM, ALGO, RUN, ID)
mean, std = x.mean(0), x.std(0)
err = mean - t[0]

# truth, mean and the individual samples are the SAME quantity -> one shared scale
field = scale(torch.cat([t, x[:N_SHOW]]).numpy())

fig, ax = plt.subplots(2, 2 + N_SHOW, figsize=(2.05 * (2 + N_SHOW), 4.6))
# One colourbar per QUANTITY, on the last panel of its group: the panels in a
# group already share a scale, and 12 identical bars only add clutter.
panel(ax[0, 0], t[0], 'ground truth', cb=False, **field)
panel(ax[0, 1], mean, f'ensemble mean (N={x.shape[0]})', cb=False, **field)
for j in range(N_SHOW):
    panel(ax[0, 2 + j], x[j], f'sample {j}', cb=(j == N_SHOW - 1), **field)

panel(ax[1, 0], err, 'error  (mean − truth)', **scale(err.numpy(), sym=True))
panel(ax[1, 1], std, 'uncertainty  (ensemble sd)', **scale(std.numpy(), signed=False))
panel(ax[1, 2], err.abs(), '|error|', **scale(err.abs().numpy(), signed=False))
# deviations of the individual samples from the mean, on one shared scale
dev = (x[:N_SHOW - 1] - mean).numpy()
ds = scale(dev, sym=True)
for j in range(N_SHOW - 1):
    panel(ax[1, 3 + j], dev[j], f'sample {j} − mean', cb=(j == N_SHOW - 2), **ds)

fig.suptitle(f'{PROBLEM}   {ALGO}/{RUN}   instance {ID}', x=0.01, ha='left', fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.95)); plt.show()

print(f'error / uncertainty  =  {float(err.pow(2).mean().sqrt() / std.mean()):.1f}x   '
      f'(1 would be calibrated)')

## 2. Methods side by side

Same instance, every method and ensemble size. Columns share a colour scale, so the
**uncertainty column is directly comparable across rows** — which is where the effect of N
shows up.

In [ ]:
PROBLEM, ID = 'navier-stokes', 0                                     # <- edit

keys = [k for k in RUNS if k[0] == PROBLEM and ID < len(RUNS[k])]

# Sort by the ENSEMBLE SIZE, which is in the run name -- sorting by the number of ids put
# N=64 after N=512 and made the sweep unreadable.
def nof(k):
    tail = k[2].split('_N')[-1]
    return (k[1], int(tail) if tail.isdigit() else 0)

rows = []
for k in sorted(keys, key=nof):
    xx, tt, _ = load(*k, ID)
    rows.append((f'{k[1]}  N={xx.shape[0]}', xx.mean(0), xx.mean(0) - tt[0], xx.std(0)))
truth = load(*keys[0], ID)[1][0]

field = scale(torch.stack([truth] + [r[1] for r in rows]).numpy())
esc = scale(np.stack([r[2].numpy() for r in rows]), sym=True)
ssc = scale(np.stack([r[3].numpy() for r in rows]), signed=False)

fig, ax = plt.subplots(len(rows), 4, figsize=(8.6, 2.15 * len(rows)), squeeze=False)
for i, (lab, m, e, s) in enumerate(rows):
    last = i == len(rows) - 1        # one colourbar per column, on the bottom row
    panel(ax[i, 0], truth, 'ground truth' if i == 0 else '', cb=last, **field)
    panel(ax[i, 1], m, 'ensemble mean' if i == 0 else '', cb=last, **field)
    panel(ax[i, 2], e, 'error' if i == 0 else '', cb=last, **esc)
    panel(ax[i, 3], s, 'uncertainty' if i == 0 else '', cb=last, **ssc)
    ax[i, 0].set_ylabel(lab, fontsize=8)
fig.suptitle(f'{PROBLEM} — instance {ID}, all methods (columns share a scale)',
             x=0.01, ha='left', fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.96)); plt.show()

## 3. Several instances

One row per test instance, to check that whatever the first view showed is typical rather
than a lucky or unlucky draw.

In [ ]:
PROBLEM, ALGO, RUN, IDS = 'navier-stokes', 'KPSH', 'gen_N512', range(5)   # <- edit

data = [load(PROBLEM, ALGO, RUN, i) for i in IDS]
field = scale(torch.cat([t for _, t, _ in data]).numpy())
esc = scale(np.stack([(x.mean(0) - t[0]).numpy() for x, t, _ in data]), sym=True)
ssc = scale(np.stack([x.std(0).numpy() for x, _, _ in data]), signed=False)

fig, ax = plt.subplots(len(data), 4, figsize=(8.6, 2.15 * len(data)), squeeze=False)
for i, (x, t, _) in enumerate(data):
    last = i == len(data) - 1        # one colourbar per column, on the bottom row
    panel(ax[i, 0], t[0], 'ground truth' if i == 0 else '', cb=last, **field)
    panel(ax[i, 1], x.mean(0), 'ensemble mean' if i == 0 else '', cb=last, **field)
    panel(ax[i, 2], x.mean(0) - t[0], 'error' if i == 0 else '', cb=last, **esc)
    panel(ax[i, 3], x.std(0), 'uncertainty' if i == 0 else '', cb=last, **ssc)
    ax[i, 0].set_ylabel(f'instance {list(IDS)[i]}', fontsize=8)
fig.suptitle(f'{PROBLEM} — {ALGO}/{RUN} across instances', x=0.01, ha='left', fontsize=10)
plt.tight_layout(rect=(0, 0, 1, 0.96)); plt.show()

## 4. Observation space

Does the reconstruction reproduce the data that was actually measured? This is the
well-posed question — many states can explain one observation, so a large error in x with a
small residual in y means the problem is under-determined, not that the method failed.

Only shown where the observation is image-like: a downsampled field for navier-stokes, and
the complex scattered field (20 transmitters x 360 receivers, shown as magnitude) for
inv-scatter. Blackhole's observation is a packed vector of amplitudes, closure phases and
their error bars, so there is nothing to look at.

**The residual has a floor**: `recon_obs` is the operator applied *with* its observation
noise, so even a perfect reconstruction differs from `observation` by one noise draw.

In [ ]:
PROBLEM, ALGO, RUN, ID = 'navier-stokes', 'KPSH', 'gen_N512', 0      # <- edit

_, _, d = load(PROBLEM, ALGO, RUN, ID)
obs, ro = d['observation'].cpu(), d['recon_obs'].cpu()

if torch.is_complex(obs):
    obs, ro, what = obs.abs(), ro.abs(), '|scattered field|'
else:
    what = 'observation'
obs, ro = obs.float(), ro.float()

if obs.squeeze().ndim != 2:
    print(f'{PROBLEM}: the observation is a packed vector of shape {tuple(obs.shape)}, '
          f'not an image — nothing to show here.')
else:
    o2, m2 = obs.squeeze(), ro.mean(0).squeeze()
    res = m2 - o2
    sc = scale(torch.stack([o2, m2]).numpy())
    fig, ax = plt.subplots(1, 3, figsize=(9.4, 3.1))
    panel(ax[0], o2, f'{what} (measured)', **sc)
    panel(ax[1], m2, 'mean of the reconstructions', **sc)
    panel(ax[2], res, 'residual', **scale(res.numpy(), sym=True))
    fig.suptitle(f'{PROBLEM} — {ALGO}/{RUN}, instance {ID}', x=0.01, ha='left', fontsize=10)
    plt.tight_layout(rect=(0, 0, 1, 0.93)); plt.show()
    print(f'relative residual of the mean: '
          f'{float(res.norm() / o2.norm()):.4f}')